<a href="https://colab.research.google.com/github/AlperYildirim1/TS-MechInterp/blob/main/Time_series_combined_pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q x-transformers

## Config

In [ ]:
import os
import math
import json
import random
import logging
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from x_transformers import Encoder
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

# 0. TARGET CONFIGURATIONS (Skipping Elec/Traffic)
DATASETS_CONFIG = {
    "weather":     {"path": "weather/weather.csv",             "split": "70_10_20",            "channels": 21,  "batch_size": 128, "d_model": 64},
    "exchange":    {"path": "exchange_rate/exchange_rate.csv", "split": "70_10_20",            "channels": 8,   "batch_size": 128, "d_model": 16},
    "etth1":       {"path": "ETT-small/ETTh1.csv",             "split":[8640, 2880, 2880],    "channels": 7,   "batch_size": 128, "d_model": 16},
    "etth2":       {"path": "ETT-small/ETTh2.csv",             "split":[8640, 2880, 2880],    "channels": 7,   "batch_size": 128, "d_model": 16},
    "ettm1":       {"path": "ETT-small/ETTm1.csv",             "split":[34560, 11520, 11520], "channels": 7,   "batch_size": 128, "d_model": 16},
    "ettm2":       {"path": "ETT-small/ETTm2.csv",             "split":[34560, 11520, 11520], "channels": 7,   "batch_size": 128, "d_model": 16},
}

PRED_LENS =[96, 192, 336, 720]
SEQ_LEN = 336
SEED = 1

EPOCHS = 80
LR = 2e-4
PATIENCE = 15
DROPOUT = 0.2

# Global Setup
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {device} (L4 Optimized)")

# Base Directories
BASE_DIR = '/content/drive/MyDrive/Master_Training'
SAVE_DIR = os.path.join(BASE_DIR, 'Checkpoints')
LOG_DIR  = os.path.join(BASE_DIR, 'Logs')
TB_DIR   = os.path.join(BASE_DIR, 'TensorBoard')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(TB_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
master_results =[]

def get_logger(run_name):
    logger = logging.getLogger(run_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fh = logging.FileHandler(os.path.join(LOG_DIR, f"{run_name}_{timestamp}.log"))
    fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(ch)
    return logger


## Architecture

In [ ]:
class TSDataset(Dataset):
    def __init__(self, data, seq_len=336, pred_len=96):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len
    def __len__(self): return len(self.data) - self.seq_len - self.pred_len + 1
    def __getitem__(self, idx):
        s = idx + self.seq_len
        return torch.FloatTensor(self.data[idx:s]), torch.FloatTensor(self.data[s:s + self.pred_len])

class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))
    def forward(self, x, mode):
        if mode == 'norm':
            self.mean = x.mean(dim=1, keepdim=True).detach()
            self.stdev = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self.mean) / self.stdev
            return x * self.affine_weight + self.affine_bias
        else:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x * self.stdev + self.mean

class AblationPatchTST(nn.Module):
    def __init__(self, seq_len=336, pred_len=96, patch_len=16, stride=8, d_model=128,
                 transformer_depth=1, dropout=0.2, n_channels=7,
                 use_rope=True, use_w_pos=False, use_glu=False):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.d_model = d_model
        self.num_patches = (seq_len - patch_len) // stride + 2
        self.use_w_pos = use_w_pos
        self.revin = RevIN(n_channels)
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_dropout = nn.Dropout(dropout)
        if self.use_w_pos:
            self.W_pos = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)
        transformer_heads = max(1, d_model // 8)
        self.transformer = Encoder(
            dim=d_model, depth=transformer_depth, heads=transformer_heads,
            attn_dim_head=16, ff_mult=2, attn_flash=True,
            rotary_pos_emb=use_rope, use_rmsnorm=True, ff_glu=use_glu,
            attn_dropout=dropout, ff_dropout=dropout
        )
        self.flatten_dim = self.num_patches * d_model
        self.head_drop = nn.Dropout(dropout)
        self.head = nn.Linear(self.flatten_dim, pred_len)

    def forward(self, x):
        x = self.revin(x, 'norm')
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)
        x = x.transpose(1, 2)
        x = F.pad(x, (0, self.stride), 'replicate')
        patches = x.unfold(-1, self.patch_len, self.stride).squeeze(1)
        raw_embed = self.patch_embed(patches)
        features = self.pos_dropout(raw_embed + self.W_pos[:, :raw_embed.shape[1], :]) if self.use_w_pos else self.pos_dropout(raw_embed)
        features = self.transformer(features)
        flattened = self.head_drop(features.reshape(features.shape[0], -1))
        out = self.head(flattened).unsqueeze(-1)
        out = out.reshape(B, C, self.pred_len).permute(0, 2, 1)
        return self.revin(out, 'denorm')


## Training Loop

In [ ]:
for ds_name, cfg in DATASETS_CONFIG.items():
    print(f"\n{'='*70}")
    print(f" PREPARING DATASET: {ds_name.upper()}")
    print(f"{'='*70}")

    # A. LOAD & SPLIT DATA
    url = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/" + cfg["path"]
    df = pd.read_csv(url)
    cols_data = df.drop(columns=[df.columns[0]]).values.astype(np.float32)
    n_total = len(cols_data)
    n_channels = cols_data.shape[1]

    if cfg["split"] == "70_10_20":
        train_end = int(n_total * 0.7)
        val_end = n_total - int(n_total * 0.2)
        test_end = n_total
    else:
        train_end = cfg["split"][0]
        val_end = train_end + cfg["split"][1]
        test_end = val_end + cfg["split"][2]

    scaler = StandardScaler()
    scaler.fit(cols_data[:train_end])
    all_data = scaler.transform(cols_data)

    train_slice = all_data[0:train_end]
    val_slice   = all_data[train_end - SEQ_LEN : val_end]
    test_slice  = all_data[val_end - SEQ_LEN : test_end]

    # B. LOOP THROUGH HORIZONS
    for pred_len in PRED_LENS:
        run_name = f"RoPE_Only_StdFFN_{ds_name}_h{pred_len}_seed{SEED}"
        logger = get_logger(run_name)
        writer = SummaryWriter(log_dir=os.path.join(TB_DIR, f"{run_name}_{timestamp}"))

        logger.info(f"\n--- ⏳ TRAINING HORIZON: {pred_len} ---")

        # Dataloaders
        g = torch.Generator().manual_seed(SEED)
        train_loader = DataLoader(TSDataset(train_slice, SEQ_LEN, pred_len), batch_size=cfg["batch_size"], shuffle=True, drop_last=True, generator=g, num_workers=0)
        val_loader   = DataLoader(TSDataset(val_slice, SEQ_LEN, pred_len), batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
        test_loader  = DataLoader(TSDataset(test_slice, SEQ_LEN, pred_len), batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
        # Model Init (Standard GELU FFN)
        model = AblationPatchTST(
            seq_len=SEQ_LEN, pred_len=pred_len, d_model=cfg["d_model"],
            transformer_depth=1, dropout=DROPOUT, n_channels=n_channels,
            use_rope=True, use_w_pos=False, use_glu=False
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        criterion = nn.MSELoss()

        best_val, no_improve = float('inf'), 0
        best_ckpt_path = os.path.join(SAVE_DIR, f"{run_name}_best.pt")

        # C. EPOCH LOOP
        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0
            pbar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}", leave=False)

            for x, y in pbar:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    loss = criterion(model(x), y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                train_loss += loss.item()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            avg_train = train_loss / len(train_loader)

            # Validation Phase
            model.eval()
            val_loss = 0
            with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    val_loss += criterion(model(x), y).item()
            avg_val = val_loss / len(val_loader)

            scheduler.step(avg_val)

            # Save Best Model & Early Stopping
            if avg_val < best_val:
                best_val = avg_val
                no_improve = 0
                tag = "🏆"
                torch.save(model.state_dict(), best_ckpt_path)
            else:
                no_improve += 1
                tag = ""

            logger.info(f"Ep {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Best: {best_val:.4f} {tag}")
            writer.add_scalars("Loss", {"train": avg_train, "val": avg_val}, epoch + 1)

            if no_improve >= PATIENCE:
                logger.info(f"⏹ Early stop triggered at epoch {epoch+1}")
                break

        # D. EVALUATION ON TEST SET
        logger.info("Loading best checkpoint for Final Testing...")
        model.load_state_dict(torch.load(best_ckpt_path))
        model.eval()

        mse_sum, mae_sum, total_samples = 0.0, 0.0, 0
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                p = model(x)
                b_size = x.size(0)
                mse_sum += F.mse_loss(p, y).item() * b_size
                mae_sum += F.l1_loss(p, y).item() * b_size
                total_samples += b_size

        final_mse = mse_sum / total_samples
        final_mae = mae_sum / total_samples

        logger.info(f"✅ FINAL {ds_name.upper()} [H={pred_len}] | Test MSE: {final_mse:.4f} | Test MAE: {final_mae:.4f}")
        writer.close()

        # Log to Master Results Array
        master_results.append({
            "Dataset": ds_name.upper(),
            "Horizon": pred_len,
            "d_model": cfg["d_model"],
            "Test_MSE": round(final_mse, 4),
            "Test_MAE": round(final_mae, 4)
        })

        # Memory Cleanup before next horizon
        del model, optimizer, train_loader, val_loader, test_loader
        torch.cuda.empty_cache()

# ==========================================
# 3. SAVE MASTER SUMMARY
# ==========================================
summary_path = os.path.join(BASE_DIR, f"Master_Training_Summary_{timestamp}.json")
with open(summary_path, 'w') as f:
    json.dump(master_results, f, indent=4)

print("\n🎉 ALL 24 MODELS SUCCESSFULLY TRAINED!")
print(f"Checkpoints saved to: {SAVE_DIR}")
print(f"Summary JSON saved to: {summary_path}")

# Display Summary Table
print("\n" + "="*50)
print(f"{'Dataset':<15} | {'Horizon':<8} | {'Test MSE':<10} | {'Test MAE'}")
print("="*50)
for r in master_results:
    print(f"{r['Dataset']:<15} | {r['Horizon']:<8} | {r['Test_MSE']:<10.4f} | {r['Test_MAE']:.4f}")
print("="*50)

In [ ]:
import os
import sys
import platform
import subprocess
import json
import datetime
import time
import torch

# Ensure psutil is installed for deep hardware scanning
try:
    import psutil
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psutil", "-q"])
    import psutil

# ==========================================
# 0. SETUP SAVE DIRECTORY
# ==========================================
LOG_DIR = '/content/drive/MyDrive/Master_Training/Logs'
os.makedirs(LOG_DIR, exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = os.path.join(LOG_DIR, f"Day_Of_Judgement_Env_Log_{timestamp}.json")

def run_cmd(cmd):
    """Safely runs a bash command and returns the output."""
    try:
        return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception as e:
        return f"Command failed: {str(e)}"

print("⚖️ Initiating Day of Judgement Environment Scan...")

# ==========================================
# 1. THE MOTHERLOAD DICTIONARY
# ==========================================
judgement_log = {
    "1_TIME_AND_SPACE": {
        "timestamp_local": datetime.datetime.now().isoformat(),
        "timestamp_utc": datetime.datetime.utcnow().isoformat(),
        "timezone": time.tzname,
    },
    "2_OS_AND_KERNEL": {
        "system": platform.system(),
        "release": platform.release(),
        "version": platform.version(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "architecture": platform.architecture(),
        "linux_distribution": run_cmd("cat /etc/os-release | grep PRETTY_NAME").replace('PRETTY_NAME=', '').strip('"')
    },
    "3_HARDWARE_CPU_RAM": {
        "cpu_model_exact": run_cmd("cat /proc/cpuinfo | grep 'model name' | uniq").replace("model name\t: ", ""),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_threads": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "ram_available_gb": round(psutil.virtual_memory().available / (1024**3), 2),
        "disk_total_gb": round(psutil.disk_usage('/').total / (1024**3), 2),
        "disk_free_gb": round(psutil.disk_usage('/').free / (1024**3), 2),
    },
    "4_HARDWARE_GPU_AND_CUDA": {
        "cuda_is_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "gpus": {},
        "nvidia_smi_dump": run_cmd("nvidia-smi") if torch.cuda.is_available() else "No GPU found"
    },
    "5_PYTORCH_DEEP_CONFIG": {
        "torch_version": torch.__version__,
        "cuda_compiled_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "nccl_version": torch.cuda.nccl.version() if torch.cuda.is_available() else "N/A",
        "is_bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        "flash_sdp_enabled": torch.backends.cuda.flash_sdp_enabled(),
        "math_sdp_enabled": torch.backends.cuda.math_sdp_enabled(),
    },
    "6_PYTHON_RUNTIME": {
        "python_executable": sys.executable,
        "python_version": sys.version,
    },
    "7_PIP_FREEZE": run_cmd(f"{sys.executable} -m pip freeze").split('\n'),
    "8_ENVIRONMENT_VARIABLES": dict(os.environ)
}

# Deep GPU Profiling
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        judgement_log["4_HARDWARE_GPU_AND_CUDA"]["gpus"][f"gpu_{i}"] = {
            "name": props.name,
            "vram_total_gb": round(props.total_memory / (1024**3), 2),
            "compute_capability": f"{props.major}.{props.minor}",
            "multi_processor_count": props.multi_processor_count,
            "l2_cache_size_mb": round(props.l2_cache_size / (1024**2), 2) if hasattr(props, 'l2_cache_size') else "Unknown"
        }

# ==========================================
# 2. SAVE TO DRIVE
# ==========================================
with open(save_path, 'w') as f:
    json.dump(judgement_log, f, indent=4)

# ==========================================
# 3. PRINT EXECUTIVE SUMMARY
# ==========================================
print(f"\n{'='*60}")
print(" 📜 EXECUTIVE ENVIRONMENT SUMMARY")
print(f"{'='*60}")
print(f"OS      : {judgement_log['2_OS_AND_KERNEL']['linux_distribution']}")
print(f"CPU     : {judgement_log['3_HARDWARE_CPU_RAM']['cpu_model_exact']} ({judgement_log['3_HARDWARE_CPU_RAM']['logical_threads']} Cores)")
print(f"RAM     : {judgement_log['3_HARDWARE_CPU_RAM']['ram_total_gb']} GB Total")
print(f"PyTorch : {judgement_log['5_PYTORCH_DEEP_CONFIG']['torch_version']} (CUDA {judgement_log['5_PYTORCH_DEEP_CONFIG']['cuda_compiled_version']})")

if torch.cuda.is_available():
    print(f"\nGPU DETECTED:")
    for i in range(judgement_log["4_HARDWARE_GPU_AND_CUDA"]["cuda_device_count"]):
        gpu = judgement_log["4_HARDWARE_GPU_AND_CUDA"]["gpus"][f"gpu_{i}"]
        print(f"  [{i}] {gpu['name']} | {gpu['vram_total_gb']} GB VRAM | Compute {gpu['compute_capability']}")
    print(f"  -> bfloat16 natively supported: {judgement_log['5_PYTORCH_DEEP_CONFIG']['is_bf16_supported']}")
else:
    print("\n⚠️ NO GPU DETECTED (CPU ONLY MODE)")

print(f"\n{'='*60}")
print(f"✅ Full 'Day of Judgement' Log saved to:")
print(f"📂 {save_path}")
print(f"   (Contains {len(judgement_log['8_ENVIRONMENT_VARIABLES'])} Env Vars & {len(judgement_log['7_PIP_FREEZE'])} Pip Packages)")
print(f"{'='*60}")